In [1]:
import os, glob, json, uuid, hashlib
import pandas as pd
from datetime import datetime

In [ ]:


MARKET = "Bravo"
INPUT_DIR_DEFAULT = "."          
BRONZE_DIR = "bronze_layer"
LOG_DIR = "logs"

run_id = str(uuid.uuid4())
run_ts = datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%SZ")
snapshot_date = datetime.now().strftime("%Y-%m-%d")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

json_files = sorted(glob.glob(os.path.join(INPUT_DIR_DEFAULT, "*.json")))
if not json_files:
    raise FileNotFoundError(f"Bu qovluqda json tapılmadı: {os.path.abspath(INPUT_DIR_DEFAULT)}")

print(f"✅ JSON tapıldı: {len(json_files)} fayl | qovluq: {os.path.abspath(INPUT_DIR_DEFAULT)}")

def money_from_wolt_int(x):
    return None if x is None else round(x / 100.0, 2)

def stable_product_hash(market, barcode, name):
    key = f"{market}|{barcode}|{name}".strip()
    return hashlib.sha256(key.encode("utf-8")).hexdigest()


all_rows = []
summary = []

for fp in json_files:
    category_name = os.path.basename(fp).replace(".json", "")

    try:
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        summary.append({"file": fp, "category": category_name, "status": "JSON_READ_ERROR", "error": str(e), "rows": 0})
        continue

    items = data.get("items", [])
    if not isinstance(items, list):
        summary.append({"file": fp, "category": category_name, "status": "NO_ITEMS_ARRAY", "error": "items is not a list", "rows": 0})
        continue

    rows_before = len(all_rows)

    for item in items:
        name = item.get("name", "") or ""

        barcode = item.get("barcode_gtin")
        barcode = "" if barcode is None else str(barcode)

        price_raw = item.get("price")
        old_price_raw = item.get("original_price")
        if old_price_raw is None:
            old_price_raw = price_raw

        price_current = money_from_wolt_int(price_raw)
        price_old = money_from_wolt_int(old_price_raw)

        purch = item.get("purchasable_balance")
        try:
            purch_int = int(purch) if purch is not None else None
        except Exception:
            purch_int = None

        availability_status = "unknown" if purch_int is None else ("available" if purch_int > 0 else "out_of_stock")

        row = {
            "scrape_run_id": run_id,
            "scrape_ts_utc": datetime.utcnow().isoformat(),
            "snapshot_date": snapshot_date,
            "market": MARKET,
            "category_raw": category_name,

            "product_name_raw": name,
            "barcode": barcode,

            "price_current_azn": price_current,
            "price_old_azn": price_old,
            "price_current_raw": price_raw,
            "price_old_raw": old_price_raw,

            "promo_flag": 1 if (price_current is not None and price_old is not None and price_current < price_old) else 0,

            # ✅ EWS üçün əsas
            "purchasable_balance": purch_int,
            "availability_status": availability_status,

            # raw əlavə info
            "unit_info_raw": item.get("unit_info"),

            # stabil product açarı
            "product_hash": stable_product_hash(MARKET, barcode, name),
        }

        all_rows.append(row)

    summary.append({"file": os.path.basename(fp), "category": category_name, "status": "OK", "items_in_json": len(items), "rows": len(all_rows) - rows_before})

df = pd.DataFrame(all_rows)
if df.empty:
    raise ValueError("items boşdur: heç bir row çıxmadı.")

df["dq_missing_barcode"] = df["barcode"].apply(lambda x: 1 if (x is None or str(x).strip() in ["", "None"]) else 0)
df["dq_invalid_price"] = df["price_current_azn"].apply(lambda x: 1 if (x is None or x <= 0) else 0)
df["dq_missing_purchasable_balance"] = df["purchasable_balance"].apply(lambda x: 1 if x is None else 0)

bronze_file = os.path.join(BRONZE_DIR, f"bravo_25fev-seher_bronze_{snapshot_date}_{run_ts}.csv")
df.to_csv(bronze_file, index=False, encoding="utf-8-sig")

log_obj = {
    "scrape_run_id": run_id,
    "run_ts_utc": datetime.utcnow().isoformat(),
    "snapshot_date": snapshot_date,
    "market": MARKET,
    "input_dir": os.path.abspath(INPUT_DIR_DEFAULT),
    "json_files_count": len(json_files),
    "rows_written": int(len(df)),
    "bronze_file": bronze_file,
    "dq": {
        "missing_barcode_rows": int(df["dq_missing_barcode"].sum()),
        "invalid_price_rows": int(df["dq_invalid_price"].sum()),
        "missing_purchasable_balance_rows": int(df["dq_missing_purchasable_balance"].sum()),
    },
    "summary_by_file": summary
}

log_file = os.path.join(LOG_DIR, f"bravo_25fev_seher_log_{snapshot_date}_{run_ts}.json")
with open(log_file, "w", encoding="utf-8") as f:
    json.dump(log_obj, f, ensure_ascii=False, indent=2)

print("✅ BRONZE saved:", bronze_file)
print("✅ LOG saved:", log_file)
print("Rows:", len(df))
print("Missing purchasable_balance:", int(df["dq_missing_purchasable_balance"].sum()))


✅ JSON tapıldı: 14 fayl | qovluq: c:\Users\Duyghu Mammadova\Desktop\ABB Python\25 fevral seher Bravo
✅ BRONZE saved: bronze_layer\bravo_25fev-seher_bronze_2026-02-25_2026-02-25T05-55-03Z.csv
✅ LOG saved: logs\bravo_25fev_seher_log_2026-02-25_2026-02-25T05-55-03Z.json
Rows: 617
Missing purchasable_balance: 0
